In [ ]:
# DIAGNOSTIC CELL: Run this to check your environment
import sys
print(f"Python Executable: {sys.executable}")
try:
    import numpy as np
    print(f"Numpy version: {np.__version__}")
    print("Numpy is working correctly!")
except ImportError:
    print("Numpy not found. Attempting to install in the current kernel...")
    %pip install numpy matplotlib seaborn pandas openpyxl
    print("Installation complete. PLEASE RESTART THE KERNEL (Kernel -> Restart) then run the cells again.")

# Dynamic Programming and Model-Free Learning: The Race Car Problem

This notebook demonstrates the full spectrum of Reinforcement Learning algorithms applied to the **Race Car MDP**:
1. **Planning (Dynamic Programming)**: Policy Evaluation, Policy Iteration, Value Iteration.
2. **Learning (Model-Free)**: Monte Carlo, TD(0), and Q-Learning.

## 1. Problem Definition

### States ($S$)
- **Cool (C)**: Engine is running at optimal temperature.
- **Warm (W)**: Engine is warm.
- **Overheated (OH)**: Engine has failed (Terminal state).

### Actions ($A$)
- **Slow (S)**: Safe speed. Lower reward, low risk.
- **Fast (F)**: High speed. Higher reward, high risk.

### Complete Model Dynamics $p(s', r | s, a)$

| Current State ($s$) | Action ($a$) | Next State ($s'$) | Reward ($r$) | Probability ($p$) |
|:---|:---|:---|:---|:---|
| Cool | Slow | Cool | +1.0 | 0.5 |
| Cool | Slow | Warm | +1.0 | 0.5 |
| Cool | Fast | Cool | +2.0 | 0.5 |
| Cool | Fast | Warm | +2.0 | 0.5 |
| Warm | Slow | Cool | +1.0 | 0.5 |
| Warm | Slow | Warm | +1.0 | 0.5 |
| Warm | Fast | Overheated | -10.0 | 1.0 |
| Overheated | *Any* | Overheated | 0.0 | 1.0 |

### Parameters
- Discount Factor $\gamma = 0.9$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass

# Constants
COOL, WARM, OVERHEATED = 0, 1, 2
STATE_NAMES = {COOL: "Cool", WARM: "Warm", OVERHEATED: "Overheated"}
SLOW, FAST = 0, 1
ACTION_NAMES = {SLOW: "Slow", FAST: "Fast"}

@dataclass
class RaceCarMDP:
    num_states: int = 3
    num_actions: int = 2
    gamma: float = 0.9
    def __post_init__(self):
        self.P = {
            COOL: { SLOW: [(0.5, COOL, 1.0, False), (0.5, WARM, 1.0, False)],
                    FAST: [(0.5, COOL, 2.0, False), (0.5, WARM, 2.0, False)] },
            WARM: { SLOW: [(0.5, COOL, 1.0, False), (0.5, WARM, 1.0, False)],
                    FAST: [(1.0, OVERHEATED, -10.0, True)] },
            OVERHEATED: { SLOW: [(1.0, OVERHEATED, 0.0, True)],
                          FAST: [(1.0, OVERHEATED, 0.0, True)] }
        }

mdp = RaceCarMDP()

## 2. Policy Evaluation (Prediction)

How good is a given policy? We iteratively compute $V_\pi(s)$.

**Initial Random Policy (Equiprobable):**

| State | Slow ($a=0$) | Fast ($a=1$) |
| :--- | :---: | :---: |
| **Cool** | 0.5 | 0.5 |
| **Warm** | 0.5 | 0.5 |
| **Overheated** | 0.5 | 0.5 |

In [ ]:
def policy_evaluation(mdp, policy, theta=1e-8):
    V = np.zeros(mdp.num_states)
    while True:
        delta = 0
        for s in range(mdp.num_states):
            v_new = 0
            for a, prob_a in enumerate(policy[s]):
                for prob, ns, r, done in mdp.P[s][a]:
                    v_new += prob_a * prob * (r + mdp.gamma * V[ns] * (not done))
            delta = max(delta, np.abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta: break
    return V

random_policy = np.ones((mdp.num_states, mdp.num_actions)) / mdp.num_actions
V_random = policy_evaluation(mdp, random_policy)
print("Value Function for Random Policy:")
for s in range(mdp.num_states): print(f"{STATE_NAMES[s]}: {V_random[s]:.4f}")

## 3. Policy Iteration (Planning)

Finding the right policy is about finding the right **Action**. We use $Q_\pi(s, a)$ as a 'scorecard' and `argmax()` to pick the winner.

$$\pi'(s) = \text{argmax}_a Q_\pi(s, a)$$

In [ ]:
def policy_iteration(mdp):
    policy = np.zeros((mdp.num_states, mdp.num_actions)); policy[:, SLOW] = 1.0
    V = np.zeros(mdp.num_states)
    p_hist, v_hist = [policy.copy()], [V.copy()]
    
    while True:
        V = policy_evaluation(mdp, policy)
        policy_stable = True
        new_p = policy.copy()
        for s in range(mdp.num_states):
            old_a = np.argmax(policy[s])
            q = [sum(p*(r + mdp.gamma*V[ns]*(not d)) for p, ns, r, d in mdp.P[s][a]) for a in range(mdp.num_actions)]
            best_a = np.argmax(q)
            if old_a != best_a:
                policy_stable = False
                new_p[s] = np.eye(mdp.num_actions)[best_a]
        v_hist.append(V.copy())
        if not policy_stable: policy = new_p.copy(); p_hist.append(policy.copy())
        else: break
    return policy, V, p_hist, v_hist

opt_p, V_opt, pi_h, v_h = policy_iteration(mdp)
print("Policy Iteration History:")
for i in range(len(pi_h)):
    print(f"\n--- Step {i} ---")
    data = [[STATE_NAMES[s], pi_h[i][s][SLOW], pi_h[i][s][FAST], v_h[i][s]] for s in range(mdp.num_states)]
    print(pd.DataFrame(data, columns=['State', 'pi(S)', 'pi(F)', 'V(s)']).to_string(index=False))
    print("(Initialization)" if i == 0 else "(Improved)")

print("\n--- FINAL CONCLUSION: OPTIMAL STRATEGY ---")
final_data = [[STATE_NAMES[s], ACTION_NAMES[np.argmax(opt_p[s])] if s != OVERHEATED else "Terminal"] for s in range(mdp.num_states)]
print(pd.DataFrame(final_data, columns=['State', 'Action']).to_string(index=False))

## 4. Value Iteration (Shortcut Planning)

Uses `max()` to jump directly to the optimal values without a full evaluation loop.

In [ ]:
def value_iteration(mdp):
    V = np.zeros(mdp.num_states)
    v_hist = [V.copy()]
    while True:
        delta = 0
        new_V = V.copy()
        for s in range(mdp.num_states):
            q = [sum(p*(r + mdp.gamma*V[ns]*(not d)) for p, ns, r, d in mdp.P[s][a]) for a in range(mdp.num_actions)]
            new_V[s] = max(q)
            delta = max(delta, abs(new_V[s] - V[s]))
        V = new_V; v_hist.append(V.copy())
        if delta < 1e-8: break
    return V, v_hist

V_vi, vi_h = value_iteration(mdp)
print("Value Iteration History (Select Sweeps):")
for i in [0, 1, 2, 5, 10, len(vi_h)-1]:
    if i >= len(vi_h): continue
    print(f"Sweep {i}: V(Cool)={vi_h[i][COOL]:.2f}, V(Warm)={vi_h[i][WARM]:.2f}")

# Part 2: Model-Free Learning (Actual Interaction)

Now we treat the rules as a **Black Box**. The agent learns by doing.

## 5. Step Function (The 'Actual' World)

In [ ]:
def step(mdp, s, a):
    outcomes = mdp.P[s][a]
    idx = np.random.choice(len(outcomes), p=[o[0] for o in outcomes])
    return outcomes[idx][1:] # (ns, r, done)

## 6. Monte Carlo Prediction
Wait for the end of the episode to calculate the Return ($G_t$).
Formula: $V(s) \leftarrow V(s) + \alpha [G_t - V(s)]$

In [ ]:
def mc_prediction(mdp, policy, episodes=5000, alpha=0.01):
    V = np.zeros(mdp.num_states)
    for i in range(episodes):
        s, traj, G = COOL, [], 0
        while True:
            a = np.random.choice(mdp.num_actions, p=policy[s])
            ns, r, done = step(mdp, s, a); traj.append((s, r))
            if done: break
            s = ns
        for s, r in reversed(traj): G = r + mdp.gamma * G; V[s] += alpha * (G - V[s])
        if (i+1) % 1000 == 0: print(f"Episode {i+1}: V(Cool)={V[COOL]:.2f}")
    return V
V_mc = mc_prediction(mdp, random_policy)

## 7. TD(0) and Q-Learning
Learn while driving using the **TD-Error**.
Q-Update: $Q(s, a) \leftarrow Q(s, a) + \alpha [r + \gamma \max Q(s', a') - Q(s, a)]$

In [ ]:
def q_learning(mdp, episodes=10000, alpha=0.1, eps=0.1):
    Q = np.zeros((mdp.num_states, mdp.num_actions))
    for i in range(episodes):
        s = COOL
        while True:
            a = np.random.randint(2) if np.random.rand() < eps else np.argmax(Q[s])
            ns, r, done = step(mdp, s, a)
            Q[s, a] += alpha * (r + mdp.gamma * np.max(Q[ns]) * (not done) - Q[s, a])
            if done: break
            s = ns
    return Q

Q_final = q_learning(mdp)
print("\n--- Q-LEARNING FINAL STRATEGY ---")
data_q = [[STATE_NAMES[s], ACTION_NAMES[np.argmax(Q_final[s])] if s != OVERHEATED else "Terminal"] for s in range(mdp.num_states)]
print(pd.DataFrame(data_q, columns=['State', 'Learned Action']).to_string(index=False))